In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable


In [0]:
%run "/Workspace/Users/gouravkumar792@gmail.com/consolidated_pipeline/1_setup (1)/utiles"

In [0]:
print(bronze_schema, silver_schema, gold_schema)

In [0]:
dbutils.widgets.text("catalog", "fmcg", "Catalog")
dbutils.widgets.text("data_source", "customers", "Data Source")

catalog = dbutils.widgets.get("catalog")
data_source = dbutils.widgets.get("data_source")

In [0]:
base_path = f"s3://project-databricks-practice/{data_source}/*.csv"
df = (
    spark.read.format("csv")
    .option("inferSchema", "true")
    .option("header", "true")
    .load(base_path)
    .withColumn("read_timestamp", F.current_timestamp())
# this below line of code to get the file name and file size for lineage and debugging purposes.
    .select("*", "_metadata.file_name", "_metadata.file_size")
)
display(df.limit(10))

In [0]:
df.write.format("delta").option("delta.enableChangeDataFeed", "true").mode("overwrite").saveAsTable(f"{catalog}.{bronze_schema}.{data_source}")

#SILVER PROCESSING

In [0]:
df_bronze = spark.sql(f"select * from {catalog}.{bronze_schema}.{data_source};")
df_bronze.show(10)

In [0]:
df_bronze.printSchema()

In [0]:
# is there any duplacte customer ids in the bronze table
df_duplicates = df_bronze.groupBy("customer_id").count().filter(F.col("count") > 1)
df_duplicates.show()

In [0]:
# Now we are dropping the duplicate records from the bronze table
print('Rows count before dropping duplicates: ', df_bronze.count())
df_silver = df_bronze.dropDuplicates(["customer_id"])
print("Rows count after dropping duplicates: ", df_silver.count())

In [0]:
df_silver = df_silver.withColumn("customer_name", F.trim(F.col("customer_name")))

df_silver.show(10)

In [0]:
display(
    df_silver.filter(F.col("customer_name") != F.trim(F.col("customer_name")))
)

In [0]:
df_silver.select('city').distinct().show()

In [0]:
# typos -> correct city names
city_mapping = {
    'Bengaluruu': 'Bengaluru',
    'Bengalore': 'Bengaluru',
    
    'Hyderabadd': 'Hyderabad',
    'Hyderbad': 'Hyderabad',
    
    'NewDheli': 'New Delhi',
    'NewDelhi': 'New Delhi',
    'NewDelhee': 'New Delhi'
}

allowed = ['Bengaluru', 'Hyderabad', 'New Delhi']

df_silver = (
    df_silver
    .replace(city_mapping, subset=["city"])
    .withColumn(
        "city", 
        F.when(F.col("city").isNull(), None)
        .when(F.col("city").isin(allowed), F.col("city"))
        .otherwise(None)       
        )
)

In [0]:
# df_silver.show()
df_silver.select('city').distinct().show()

In [0]:
# Title case fix of the customer name
df_silver = df_silver.withColumn(
    "customer_name",
    F.when(F.col("customer_name").isNull(), None)
    .otherwise(F.initcap("customer_name"))
)

df_silver.select("customer_id", "city").filter(F.col("city").isNull()).show()

In [0]:
customer_city_fix = {
     789403: "New Delhi",
     789420: "Bangaluru",
     789521: "Hyderabad",
     789603: "Hyderabad"
}

df_fix = spark.createDataFrame(
    [(k, v) for k, v in customer_city_fix.items()],
    ["customer_id", "fixed_city"]
)
df_fix.show()

In [0]:
df_silver = (
    df_silver.join(df_fix, on="customer_id", how="left")
    .withColumn(
        "city",
        F.coalesce("city", "fixed_city")
    )
    .drop("fixed_city")
)


In [0]:
df_silver.show()

In [0]:
df_silver = df_silver.withColumn("customer_id", F.col("customer_id").cast("string"))
print(df_silver.printSchema())

In [0]:
# for the parent company customer gold table there is no column city so we merge the customer name and city
# and harcode the country, platform and channel column 
df_silver = (
    df_silver
    .withColumn("customer",
                F.concat_ws("-", "customer_name", F.coalesce(F.col("city"), F.lit("Unknown")))
                )
    .withColumn("market", F.lit("India"))
    .withColumn("platform", F.lit("Sports Bar"))
    .withColumn("channel", F.lit("Acquisition"))
)
display(df_silver.limit(10))

In [0]:
df_silver.write.format("delta").mode("overwrite").option("delta.enableChangeDataFeed", "true") \
.option("mergeSchema", "true") \
.saveAsTable(f'{catalog}.{silver_schema}.{data_source}')

#Gold Processing

In [0]:
df_silver = spark.sql(f"SELECT * FROM {catalog}.{silver_schema}.{data_source};")

df_gold = df_silver.select("customer_id", "customer_name", "city", "customer", "market", "platform", "channel")

In [0]:
df_gold.write.format("delta").option("delta.enableChangeDataFeed", "true").mode("overwrite") \
    .saveAsTable(f"{catalog}.{gold_schema}.sb_dm_{data_source}")

In [0]:
delta_table = DeltaTable.forName(spark, "fmcg.gold.dim_customers")
df_child_customers = spark.table("fmcg.gold.sb_dm_customers").select(
    F.col("customer_id").alias("customer_code"),
    "customer",
    "market",
    "platform",
    "channel"
)

In [0]:
delta_table.alias("target").merge(
    source=df_child_customers.alias("source"),
    condition="target.customer_code = source.customer_code"
).whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()